# LAB | Extractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

# Install Dependencies

In [1]:
%pip install -q transformers==4.46.3 datasets==4.5.0 pinecone==5.4.2 sentence-transformers==3.3.1 huggingface_hub==0.26.5 python-dotenv pandas tqdm

In [2]:
# If any of these libraries were already imported before installation,
# restart the kernel manually before continuing with the next section.
print("Dependencies installed. Continue below with a fresh kernel if needed.")

Dependencies installed. Continue below with a fresh kernel if needed.


Run the installation cell once. If Jupyter or Colab requests a restart, restart the kernel manually and continue from **Load Dataset**. The notebook does not force a restart or delete cached models.

# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [3]:
import os

# This lab uses PyTorch; avoid importing optional TensorFlow/Keras packages.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("rajpurkar/squad", split="train").to_pandas()

In [4]:
# Keep one row for every distinct context passage.
df = df[["title", "context"]].copy()
df = df.drop_duplicates(subset="context").reset_index(drop=True)
assert df["context"].is_unique
df

,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,The university is the major seat of the Congre...
3,University_of_Notre_Dame,The College of Engineering was established in ...
4,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...
...,...,...
18886,Kathmandu,"Institute of Medicine, the central college of ..."
18887,Kathmandu,Football and Cricket are the most popular spor...
18888,Kathmandu,The total length of roads in Nepal is recorded...
18889,Kathmandu,The main international airport serving Kathman...


# Pinecone Setup

Pinecone is a managed vector database that lets us store context embeddings and run fast semantic search over them. In this section, we set up the connection to Pinecone using an API key and create the index that will hold our context embeddings.

In [5]:
import os
import getpass
from dotenv import load_dotenv

# Load local credentials without overwriting existing environment variables.
load_dotenv()

#
# For this notebook, you'll need a Pinecone API key, 
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager 
# (the key icon in the left sidebar). 
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')

<br>

Once we have the API key, we can initialize the connection:

In [6]:
from pinecone import Pinecone, ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(
    api_key = api_key
)

Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [7]:
import time

index_name = "question-answering"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec,
    )

# A new serverless index may take a moment to become ready.
deadline = time.monotonic() + 180
while True:
    description = pc.describe_index(index_name)
    if description.dimension != 384 or description.metric != "cosine":
        raise ValueError("The existing index must use 384 dimensions and cosine similarity.")
    if description.status["ready"]:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("The Pinecone index was not ready within 180 seconds.")
    time.sleep(2)

index = pc.Index(index_name)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [8]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
retriever = SentenceTransformer(
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1", device=device
)
assert retriever.get_sentence_embedding_dimension() == 384
retriever

W1027 15:43:36.959000 23728 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [9]:
from hashlib import sha256
from tqdm.auto import tqdm

batch_size = 64
namespace = "squad-train"

for i in tqdm(range(0, len(df), batch_size)):
    i_end = min(i + batch_size, len(df))
    batch = df.iloc[i:i_end]
    emb = retriever.encode(
        batch["context"].tolist(),
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()
    meta = batch[["title", "context"]].to_dict(orient="records")
    # Stable IDs let reruns update existing passages instead of duplicating them.
    ids = [sha256(c.encode("utf-8")).hexdigest() for c in batch["context"]]
    to_upsert = list(zip(ids, emb, meta))
    index.upsert(vectors=to_upsert, namespace=namespace)

# Pinecone updates are eventually consistent; wait for the expected count.
deadline = time.monotonic() + 180
while True:
    stats = index.describe_index_stats()
    vector_count = stats.namespaces.get(namespace, {}).get("vector_count", 0)
    if vector_count >= len(df):
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Not all uploaded passages are visible yet; retry the stats check.")
    time.sleep(2)
stats

  0%|          | 0/296 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'squad-train': {'vector_count': 18891}},
 'total_vector_count': 18891}

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [10]:
from transformers import pipeline

model_name = 'deepset/electra-base-squad2'
# load the reader model into a question-answering pipeline

device_id = 0 if torch.cuda.is_available() else -1
reader = pipeline(tokenizer=model_name, model=model_name, task='question-answering', device=device_id)

reader

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

C:\Users\zinam\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zinam\.cache\huggingface\hub\models--deepset--electra-base-squad2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [11]:
def get_context(question, top_k):
    """Retrieve the most similar SQuAD passages for a nonempty question."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question must be a nonempty string")
    if not isinstance(top_k, int) or isinstance(top_k, bool) or top_k < 1:
        raise ValueError("top_k must be a positive integer")
    xq = retriever.encode(question, normalize_embeddings=True).tolist()
    xc = index.query(
        vector=xq,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
    )
    c = [match["metadata"]["context"] for match in xc["matches"]]
    return c

In [12]:
def extract_answer(question, context):
    """Return answers ranked by reader confidence, including no-answer results."""
    results = []
    for c in context:
        answer = reader(question=question, context=c, handle_impossible_answer=True)
        answer["context"] = c
        results.append(answer)
    return sorted(results, key=lambda x: x["score"], reverse=True)

In [13]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [14]:
extract_answer(question, context)

[{'score': 0.9999852180480957,
  'start': 20,
  'end': 33,
  'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.'}]

The expected answer is *691,000 bbl/d*, if the relevant passage is retrieved. The reader score expresses model confidence for this passage; it is not a measured accuracy and does not establish that an answer is correct. Let's try a few more queries.

In [15]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'score': 0.9999276399612427,
  'start': 64,
  'end': 79,
  'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, Hurley and Chen developed the idea for YouTube during the early months of 2005, after they had experienced difficulty sharing videos that had been shot at a dinner party at Chen\'s apartment in San Francisco. Karim did not attend the party and denied that it had occurred, but Chen commented that the idea that YouTube was founded after a dinner party "was probably very strengthened by marketing ideas around creating a story that was very digestible".'}]

In [16]:
question = "What is Albert Einstein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'score': 0.9851928353309631,
  'start': 29,
  'end': 86,
  'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity and general relativity. He also made important contributions to statistical mechanics, especially his mathematical treatment of Brownian motion, his resolution of the paradox of specific heats, and his connection of fluctuations and dissipation. Despite his reservations about its interpretation, Einstein also made contributions to quantum mechanics and, indirectly, quantum field theory, primarily through his theoretical studies of the photon.'}]

Let's run another question. This time for top 3 context passages from the retriever.

In [17]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'score': 0.9998037815093994,
  'start': 71,
  'end': 80,
  'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving orbit, Armstrong and Aldrin transferred into the Lunar Module, named Eagle, and after a landing gear inspection by Collins remaining in the Command/Service Module Columbia, began their descent. After overcoming several computer overload alarms caused by an antenna switch left in the wrong position, and a slight downrange error, Armstrong took over manual flight control at about 180 meters (590 ft), and guided the Lunar Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 (3:17:04 pm CDT). The first humans on the Moon would wait another six hours before they ventured out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the first human to set foot on the Moon.'},
 {'score': 0.6958646774291992,
  'start': 240,
  'end': 246,
  'answer': 'Aldrin',
  'context': 'The first step was witnessed by at

<br>

### 🚀 Challenge

Try adding a few more questions of your own in new cells below (reusing `get_context` and `extract_answer`). Experiment with different `top_k` values and observe:

- Does the retriever always return a context that actually contains the answer?
- How does the reader's confidence score change with harder or more ambiguous questions?
- What happens with questions that have no good answer in the dataset?

In [18]:
import pandas as pd

challenge_questions = [
    ("Who founded the University of Notre Dame?", "straightforward"),
    ("When was the university founded?", "ambiguous: university unspecified"),
    ("What is the name of the first city built on Mars?", "unanswerable"),
]

challenge_rows = []
for question, kind in challenge_questions:
    for top_k in (1, 3, 5):
        contexts = get_context(question, top_k=top_k)
        answers = extract_answer(question, contexts)
        # Inspect every candidate, including its source passage.
        for rank, answer in enumerate(answers, start=1):
            challenge_rows.append({
                "question": question,
                "kind": kind,
                "top_k": top_k,
                "reader_rank": rank,
                "answer": answer["answer"] or "[no answer]",
                "confidence": answer["score"],
                "context": answer["context"],
            })

challenge_results = pd.DataFrame(challenge_rows)
challenge_results

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,question,kind,top_k,reader_rank,answer,confidence,context
0,Who founded the University of Notre Dame?,straightforward,1,1,[no answer],0.999840,All of Notre Dame's undergraduate students are...
1,Who founded the University of Notre Dame?,straightforward,3,1,[no answer],0.999951,The University of Notre Dame du Lac (or simply...
2,Who founded the University of Notre Dame?,straightforward,3,2,[no answer],0.999875,The Notre Dame football team has a long histor...
3,Who founded the University of Notre Dame?,straightforward,3,3,[no answer],0.999840,All of Notre Dame's undergraduate students are...
4,Who founded the University of Notre Dame?,straightforward,5,1,[no answer],0.999951,The University of Notre Dame du Lac (or simply...
5,Who founded the University of Notre Dame?,straightforward,5,2,[no answer],0.999889,"Besides its prominence in sports, Notre Dame i..."
6,Who founded the University of Notre Dame?,straightforward,5,3,[no answer],0.999875,The Notre Dame football team has a long histor...
7,Who founded the University of Notre Dame?,straightforward,5,4,[no answer],0.999840,All of Notre Dame's undergraduate students are...
8,Who founded the University of Notre Dame?,straightforward,5,5,[no answer],0.999815,"Since 2005, Notre Dame has been led by John I...."
9,When was the university founded?,ambiguous: university unspecified,1,1,1952,0.956866,A Royal Charter in 1952 upgraded University Co...


### Challenge discussion
Inspect the candidate answers and their contexts in the table above. Retrieval ranks semantic similarity, so even the nearest passage may not contain the answer. Increasing `top_k` gives the reader more chances to find a useful passage, but also introduces irrelevant candidates.

The university question is ambiguous because it does not name a university. An answer may fit a retrieved passage while failing to resolve the intended question. Compare its scores across passages; confidence need not decrease for ambiguous questions or increase with `top_k`.

The Mars question has no factual answer in SQuAD. With `handle_impossible_answer=True`, the reader can return an empty answer, shown as `[no answer]`. It may still extract a misleading span with high confidence. Verify the source passage rather than treating confidence as proof. These are expected behaviors to investigate, not claims about unexecuted results.


### Verified execution

All code cells completed locally with GPU acceleration and 18,891 passages indexed in Pinecone. The oil question returned `691,000 bbl/d`; the Einstein question returned his theories of special and general relativity; the highest-ranked Moon answer was `Armstrong`. The YouTube question returned `Hurley and Chen` despite requesting first names, illustrating that a high reader score does not guarantee a complete answer to the question. Lower-ranked Moon candidates also included incorrect answers, so inspect the passage and ranking.


<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [19]:
# To delete the Pinecone index, 
# uncomment the line below + run this code cell

# pc.delete_index(index_name)